In [ ]:
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('car_prediction_data.csv')

df['Car_Age'] = datetime.datetime.now().year - df['Year']
df.drop(['Car_Name', 'Year'], axis=1, inplace=True)

X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


categorical_features = ['Fuel_Type', 'Seller_Type', 'Transmission']
numerical_features = ['Present_Price', 'Kms_Driven', 'Car_Age', 'Owner']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

gb_model = GradientBoostingRegressor(random_state=42)

model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', gb_model)])


param_grid = {
    'regressor__n_estimators': [100, 200, 300],       
    'regressor__learning_rate': [0.01, 0.05, 0.1],   
    'regressor__max_depth': [3, 4, 5],                
    'regressor__subsample': [0.7, 0.8, 1.0]           
}

search = RandomizedSearchCV(
    model_pipeline, 
    param_distributions=param_grid, 
    n_iter=20, 
    cv=5, 
    verbose=1, 
    random_state=42, 
    n_jobs=-1
)

print("Training in progress... (This might take a moment)")
search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print("y actual:",y_test[:5])
print("y predicted:",y_pred[:5])

print("\n--- Model Performance Report ---")
print(f"Best Parameters: {search.best_params_}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE (Mean Absolute Error): {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE (Root Mean Squared Error): {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

Training in progress... (This might take a moment)
Fitting 5 folds for each of 20 candidates, totalling 100 fits
y actual: 177     0.35
289    10.11
228     4.95
198     0.15
60      6.95
Name: Selling_Price, dtype: float64
y predicted: [ 0.41147267 11.46960568  5.0189359   0.24696773  7.58669037]

--- Model Performance Report ---
Best Parameters: {'regressor__subsample': 0.8, 'regressor__n_estimators': 200, 'regressor__max_depth': 4, 'regressor__learning_rate': 0.05}
R2 Score: 0.9728
MAE (Mean Absolute Error): 0.5241
RMSE (Root Mean Squared Error): 0.7917
